In [1]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_network_new_kernels import CNNModel
from networks.cnn_network_new_kernels import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
  -> ny bästa modell sparad till ../models/convolution_model_kernel.pth
Epoch   0 | train: 1.1271 | val: 0.6582 | acc: 67.23% | AUC: 0.720  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_kernel.pth
Epoch   1 | train: 0.6114 | val: 0.5554 | acc: 81.84% | AUC: 0.846  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_kernel.pth
Epoch   2 | train: 0.4939 | val: 0.3724 | acc: 92.09% | AUC: 0.875  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_kernel.pth
Epoch   3 | train: 0.4466 | val: 0.4136 | acc: 85.79% | AUC: 0.890  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_kernel.pth
Epoch   4 | train: 0.4375 | val: 0.4422 | acc: 81.02% | AUC: 0.896  | LR: 0.001
Epoch   5 | train: 0.4236 | val: 0.3151 | acc: 94.65% | AUC: 0.893  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_kernel.pt

In [2]:
con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
  -> ny bästa modell sparad till ../models/convolution_model_kernel_fold1.pth
Epoch   0 | train: 0.9336 | val: 0.6504 | acc: 85.56% | AUC: 0.654  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_kernel_fold1.pth
Epoch   1 | train: 0.5616 | val: 0.5155 | acc: 81.80% | AUC: 0.820  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_kernel_fold1.pth
Epoch   2 | train: 0.4608 | val: 0.4542 | acc: 84.31% | AUC: 0.847  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_kernel_fold1.pth
Epoch   3 | train: 0.4375 | val: 0.4800 | acc: 91.41% | AUC: 0.853  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolution_model_kernel_fold1.pth
Epoch   4 | train: 0.4338 | val: 0.4419 | acc: 88.90% | AUC: 0.859  | LR: 0.001
  -> ny bästa modell sparad till ../models/convolutio

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.108727,0.185391,95.636026,0.984774,0.961783,0.918519,1812,72,22,248
1,2,0.111269,0.116883,95.868152,0.991443,0.958599,0.959259,1806,78,11,259
2,3,0.086874,0.192857,96.146704,0.979056,0.969231,0.907063,1827,58,25,244
3,4,0.133191,0.199793,92.100372,0.982238,0.914498,0.966543,1722,161,9,260
4,5,0.081580,0.143839,95.078923,0.988603,0.952255,0.940520,1795,90,16,253
5,6,0.082911,0.124821,96.518106,0.987138,0.967639,0.947955,1824,61,14,255
6,7,0.097655,0.127379,96.703807,0.985037,0.973475,0.921933,1835,50,21,248
7,8,0.097932,0.194156,96.377148,0.979412,0.974522,0.888476,1836,48,30,239
8,9,0.109906,0.161018,93.726766,0.982783,0.936272,0.944238,1763,120,15,254
9,10,0.108222,0.129640,96.193129,0.988043,0.968170,0.918216,1825,60,22,247


In [3]:
test_rows = test_rows[test_rows['label'].isin([0,1])]
result = CNN.predict(test_rows)
_ = evaluate(result)

KeyError: 'prediction'